# Geospatial Layers — Riverine State Overview

Maps three contextual layers for the selected state's riverine LGAs: Google historical inundation extent, NiHSA at-risk community locations by flood depth zone, and WorldPop 2020 population estimates. Set `STATE` below to switch between Adamawa and Benue state.

In [ ]:
import pandas as pd
import ocha_stratus as stratus
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from shapely.geometry import box

from src.datasources import hydrosheds, worldpop
from src.constants import STATE_CONFIG

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

## State configuration

Set `STATE` to `"Benue"` or `"Adamawa"`.

In [ ]:
STATE = "Benue"  # or "Adamawa"
cfg = STATE_CONFIG[STATE]

## Load geolayers

In [ ]:
gdf_lga = stratus.codab.load_codab_from_blob("NGA", admin_level=2)
gdf_rivers = hydrosheds.load_selected_rivers()

gdf_state = gdf_lga[gdf_lga[cfg["adm1_col"]] == cfg["adm1_val"]].copy()
gdf_intersecting = gdf_state[gdf_state["ADM2_PCODE"].isin(cfg["lga_pcodes"])]
gdf_non_intersecting = gdf_state.drop(index=gdf_intersecting.index)

gdf_benue_river = gdf_rivers[
    (gdf_rivers.geometry.centroid.x > cfg["river_x_min"]) & (gdf_rivers.geometry.centroid.y < 10)
].copy()
minx, _, maxx, _ = gdf_state.total_bounds
gdf_river_clipped = gpd.clip(gdf_benue_river, box(minx, -90, maxx, 90))

## Load contextual datasets

In [ ]:
# Google historical inundation
gdf_inundation = stratus.load_geoparquet_from_blob(
    "ds-aa-nga-flooding/processed/google_inundation_history/combined_nga.parquet"
)
gdf_inundation_state = gpd.clip(gdf_inundation, gdf_state.union_all())
print(f"Inundation polygons: {len(gdf_inundation_state)}")

# NiHSA at-risk communities
df_risk = stratus.load_csv_from_blob(
    "ds-aa-nga-flooding/raw/AA-nigeria_data/NiHSA/AFO_communities_atrisk_2026.csv"
)
df_risk["depth_zone"] = df_risk["depth_zone"].str.strip().str.capitalize()
gdf_risk = gpd.GeoDataFrame(
    df_risk,
    geometry=gpd.points_from_xy(df_risk["lon"], df_risk["lat"]),
    crs=4326,
)
gdf_risk_state = gdf_risk[gdf_risk.within(gdf_state.union_all())]
print(f"At-risk communities: {len(gdf_risk_state):,}")

# WorldPop 2020 population (1km), clipped to state
da_wp = worldpop.load_worldpop_from_stac("NGA", year=2020, resolution="1km")
da_wp_state = da_wp.rio.clip(gdf_state.geometry, crs=4326, all_touched=True)
da_wp_state = da_wp_state.where(da_wp_state > 0)

## Google inundation history

Polygons representing areas with historical flood inundation from the Google Flood Hub dataset, clipped to the state boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

gdf_non_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5)
gdf_intersecting.plot(ax=ax, color="#E8F4FD", edgecolor="white", linewidth=0.5)
gdf_inundation_state.plot(ax=ax, facecolor="#F2645A", edgecolor="none", alpha=0.5, zorder=3)
gdf_river_clipped.plot(ax=ax, color="#1E5A8E", linewidth=1.0, zorder=4)

ax.set_axis_off()
ax.set_title(f"Google inundation history — {STATE} State", fontsize=13, pad=12)
ax.legend(
    handles=[Patch(facecolor="#F2645A", alpha=0.5, label="Inundation history")],
    frameon=False, fontsize=9, loc="lower left",
)

plt.tight_layout()
fig.savefig(FIGURES_DIR / f"{STATE.lower()}_inundation_map.png", bbox_inches="tight", dpi=150)
plt.show()

## NiHSA at-risk communities

Locations of communities identified as flood-prone by the Nigeria Hydrological Services Agency (NiHSA), coloured by depth zone (Low / Medium / High).

In [ ]:
ZONE_ORDER  = ["Low", "Medium", "High"]
ZONE_COLORS = {"Low": "#4A90E2", "Medium": "#F77F00", "High": "#E63946"}

fig, ax = plt.subplots(figsize=(10, 8))

gdf_non_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5)
gdf_intersecting.plot(ax=ax, color="#E8F4FD", edgecolor="white", linewidth=0.5)
gdf_river_clipped.plot(ax=ax, color="#1E5A8E", linewidth=1.0, zorder=3)
for zone in ZONE_ORDER:
    sub = gdf_risk_state[gdf_risk_state["depth_zone"] == zone]
    sub.plot(ax=ax, color=ZONE_COLORS[zone], markersize=4, alpha=0.75, zorder=4)

ax.set_axis_off()
ax.set_title(f"NiHSA at-risk communities — {STATE} State (2026)", fontsize=13, pad=12)
ax.legend(
    handles=[Patch(facecolor=ZONE_COLORS[z], alpha=0.85, label=f"{z} depth") for z in ZONE_ORDER],
    frameon=False, fontsize=9, loc="lower left", title="Depth zone", title_fontsize=9,
)

plt.tight_layout()
fig.savefig(FIGURES_DIR / f"{STATE.lower()}_risk_communities_map.png", bbox_inches="tight", dpi=150)
plt.show()

print(gdf_risk_state["depth_zone"].value_counts().reindex(ZONE_ORDER))

## WorldPop population (2020)

WorldPop 2020 unconstrained population estimates at 1 km resolution, clipped to the state boundary. Zero and no-data values are masked out.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

gdf_non_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5, zorder=1)
gdf_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5, zorder=1)

xlim, ylim = ax.get_xlim(), ax.get_ylim()

da_wp_state.plot(
    ax=ax,
    cmap="YlOrRd",
    robust=True,
    add_colorbar=True,
    cbar_kwargs={"label": "Population per km²", "shrink": 0.5, "pad": 0.02},
    zorder=2,
)

ax.set_xlim(xlim); ax.set_ylim(ylim)
gdf_state.boundary.plot(ax=ax, color="white", linewidth=0.5, zorder=3)
gdf_river_clipped.plot(ax=ax, color="#1E5A8E", linewidth=1.0, zorder=4)

ax.set_axis_off()
ax.set_title(f"WorldPop population density — {STATE} State (2020, 1km)", fontsize=13, pad=12)

plt.tight_layout()
fig.savefig(FIGURES_DIR / f"{STATE.lower()}_worldpop_map.png", bbox_inches="tight", dpi=150)
plt.show()